In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

load_dotenv("../.env")  # Loads GOOGLE_API_KEY from the root project folder

@tool
def check_chamber_temp(chamber_id: str) -> str:
    """Returns the current temperature of a semiconductor etch chamber."""
    return f"Chamber {chamber_id} is running at 65.2C (within spec)."

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
llm_with_tools = llm.bind_tools([check_chamber_temp])

response = llm_with_tools.invoke("Check the temperature in Chamber B2.")
print("Tool calls generated by Gemini:", response.tool_calls)


Tool calls generated by Gemini: [{'name': 'check_chamber_temp', 'args': {'chamber_id': 'B2'}, 'id': 'call_8147966984320287802', 'type': 'tool_call'}]


In [2]:
import joblib
import numpy as np

bundle = joblib.load("../models/secom_yield_bundle.joblib")
model = bundle["model"]
explainer = bundle["explainer"]
X_test = bundle["X_test_clean"]
y_test = bundle["y_test"]

shap_values = explainer.shap_values(X_test)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top10_idx = np.argsort(mean_abs_shap)[::-1][:10]

print("Top 10 Global Root-Cause Sensors:")
for rank, idx in enumerate(top10_idx, 1):
    col = X_test.columns[idx]
    median_val = X_test[col].median()
    std_val = X_test[col].std()
    print(f"  {rank}. {col} (median={median_val:.3f}, std={std_val:.3f})")

probs = model.predict_proba(X_test)[:, 1]
high_risk = X_test[(y_test == 1) & (probs > 0.5)]
print(f"\nHigh-risk failing test wafers found: {len(high_risk)}")
for wid in high_risk.index[:5]:
    p = model.predict_proba(X_test.loc[[wid]])[0, 1]
    print(f"  - Wafer ID {wid}: Fail Prob = {p:.2%}")


D:\Git\semiconductor_yield_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Top 10 Global Root-Cause Sensors:
  1. Sensor_103 (median=-0.010, std=0.003)
  2. Sensor_33 (median=8.765, std=1.506)
  3. Sensor_59 (median=0.731, std=7.863)
  4. Sensor_477 (median=5.220, std=4.196)
  5. Sensor_319 (median=8.382, std=2.075)
  6. Sensor_81 (median=-0.019, std=0.019)
  7. Sensor_577 (median=13.898, std=14.096)
  8. Sensor_31 (median=3.431, std=0.543)
  9. Sensor_21 (median=-5533.875, std=584.791)
  10. Sensor_213 (median=0.058, std=0.082)

High-risk failing test wafers found: 1
  - Wafer ID 11: Fail Prob = 52.86%
